In [2]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

print("Environment is working!")

Environment is working!


In [ ]:
# Stock Return Prediction Using Machine Learning

## Objective

Predict whether a stock will have a positive return over the next 5 trading days using historical price, volume, and technical indicators.

## Target

- `1` → future 5-day return > 0
- `0` → future 5-day return ≤ 0

## Models

- Logistic Regression
- Random Forest
- Gradient Boosting
- SVM
- KNN

## Evaluation

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

In [3]:
ticker = "RELIANCE.NS"

data = yf.download(
    ticker,
    start="2018-01-01",
    end="2026-09-01",
    auto_adjust=True
)

data.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS,RELIANCE.NS
Date,,,,,
2018-01-01,400.015289,405.709363,399.025968,405.709363,9453202
2018-01-02,400.630829,404.324304,398.542264,401.444276,9499419
2018-01-03,402.235779,407.160403,401.466300,406.720688,13507800
2018-01-04,404.654114,405.313671,402.631502,403.708761,9008932
2018-01-05,405.951233,407.556144,404.632147,405.313689,7441284


In [4]:
print("Rows:", len(data))
print("Columns:", data.columns)

Rows: 2143
Columns: MultiIndex([( 'Close', 'RELIANCE.NS'),
            (  'High', 'RELIANCE.NS'),
            (   'Low', 'RELIANCE.NS'),
            (  'Open', 'RELIANCE.NS'),
            ('Volume', 'RELIANCE.NS')],
           names=['Price', 'Ticker'])


In [5]:
data = data.xs("RELIANCE.NS", level="Ticker", axis=1)

data.head()

Price,Close,High,Low,Open,Volume
Date,,,,,
2018-01-01,400.015289,405.709363,399.025968,405.709363,9453202
2018-01-02,400.630829,404.324304,398.542264,401.444276,9499419
2018-01-03,402.235779,407.160403,401.466300,406.720688,13507800
2018-01-04,404.654114,405.313671,402.631502,403.708761,9008932
2018-01-05,405.951233,407.556144,404.632147,405.313689,7441284


In [6]:
data.info()


<class 'pandas.DataFrame'>
DatetimeIndex: 2143 entries, 2018-01-01 to 2026-08-31
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   2143 non-null   float64
 1   High    2143 non-null   float64
 2   Low     2143 non-null   float64
 3   Open    2143 non-null   float64
 4   Volume  2143 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 100.5 KB


In [7]:
data.isna().sum()

Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [8]:
tickers = [
    "RELIANCE.NS",
    "TCS.NS",
    "INFY.NS",
    "HDFCBANK.NS",
    "ICICIBANK.NS",
    "SBIN.NS",
    "BHARTIARTL.NS",
    "ITC.NS",
    "LT.NS",
    "AXISBANK.NS",
    "MARUTI.NS",
    "SUNPHARMA.NS",
    "HINDUNILVR.NS",
    "TITAN.NS",
    "ASIANPAINT.NS",
    "BAJFINANCE.NS",
    "ADANIENT.NS",
    "WIPRO.NS",
    "NTPC.NS",
    "POWERGRID.NS"
]

prices = yf.download(
    tickers,
    start="2018-01-01",
    end="2026-09-01",
    auto_adjust=True,
    group_by="ticker"
)

print("Downloaded successfully")
print("Shape:", prices.shape)

[*********************100%***********************]  20 of 20 completed

Downloaded successfully
Shape: (2143, 100)


In [9]:
prices.head()

Ticker     RELIANCE.NS                                                \
Price             Open        High         Low       Close    Volume   
Date                                                                   
2018-01-01  405.709363  405.709363  399.025968  400.015289   9453202   
2018-01-02  401.444276  404.324304  398.542264  400.630829   9499419   
2018-01-03  406.720688  407.160403  401.466300  402.235779  13507800   
2018-01-04  403.708761  405.313671  402.631502  404.654114   9008932   
2018-01-05  405.313689  407.556144  404.632147  405.951233   7441284   

Ticker            LT.NS                                                  ...  \
Price              Open         High          Low        Close   Volume  ...   
Date                                                                     ...   
2018-01-01  1107.894477  1119.316070  1103.413720  1107.630859  2048058  ...   
2018-01-02  1113.165950  1115.713865  1091.948125  1098.010376  1512324  ...   
2018-01-03  1103.413821  1128.058041  1099.548029  1120.458374  2720736  ...   
2018-01-04  1128.936662  1159.731080  1124.675551  1155.953125  4546456  ...   
2018-01-05  1164.123799  1167.594163  1149.187871  1155.250122  2418902  ...   

Ticker         NTPC.NS                                                \
Price             Open        High         Low       Close    Volume   
Date                                                                   
2018-01-01  105.260660  105.439018  104.458052  104.963394   6689762   
2018-01-02  105.112032  106.568622  104.963404  106.360535  10171537   
2018-01-03  106.717230  107.192853  105.082293  105.260651  12409998   
2018-01-04  105.498462  105.765998  104.963390  105.290375   8712321   
2018-01-05  105.409290  105.974092  104.933666  105.528198  12431070   

Ticker        MARUTI.NS                                                 
Price              Open         High          Low        Close  Volume  
Date                                                                    
2018-01-01  8962.644496  8999.418092  8853.059003  8873.376953  426354  
2018-01-02  8916.678431  8934.697135  8756.713275  8783.419922  451227  
2018-01-03  8822.906930  8823.826270  8643.635611  8657.057617  658281  
2018-01-04  8687.763200  8722.698121  8611.642026  8629.155273  562919  
2018-01-05  8695.118438  8763.149603  8653.334252  8672.916016  732512  

[5 rows x 100 columns]

In [10]:
import os

print("Current folder:")
print(os.getcwd())

Current folder:
c:\Users\Manan Bhayani\OneDrive\Desktop\ml project 1\notebooks


In [11]:
import os

print(os.getcwd())

c:\Users\Manan Bhayani\OneDrive\Desktop\ml project 1\notebooks


In [12]:
print(prices.shape)

(2143, 100)


In [13]:
prices.to_csv("../data/raw/stock_prices_20.csv")

print("Raw dataset saved successfully!")

Raw dataset saved successfully!


In [14]:
prices_long = prices.stack(level=0, future_stack=True)

prices_long.head()

Price                           Open         High          Low        Close  \
Date       Ticker                                                             
2018-01-01 RELIANCE.NS    405.709363   405.709363   399.025968   400.015289   
           LT.NS         1107.894477  1119.316070  1103.413720  1107.630859   
           TCS.NS        1078.194531  1083.219111  1059.181500  1063.442383   
           SBIN.NS        281.521387   283.470097   277.623940   278.349060   
           POWERGRID.NS    72.509389    73.492939    72.509389    72.928307   

Price                      Volume  
Date       Ticker                  
2018-01-01 RELIANCE.NS    9453202  
           LT.NS          2048058  
           TCS.NS         1351760  
           SBIN.NS       12182400  
           POWERGRID.NS   6388017

In [15]:
print(prices_long.shape)

(42860, 5)


In [16]:
print(prices_long.columns)

Index(['Open', 'High', 'Low', 'Close', 'Volume'], dtype='str', name='Price')


In [17]:
prices_long = prices_long.reset_index()

prices_long.head()

Price,Date,Ticker,Open,High,Low,Close,Volume
0,2018-01-01,RELIANCE.NS,405.709363,405.709363,399.025968,400.015289,9453202
1,2018-01-01,LT.NS,1107.894477,1119.316070,1103.413720,1107.630859,2048058
2,2018-01-01,TCS.NS,1078.194531,1083.219111,1059.181500,1063.442383,1351760
3,2018-01-01,SBIN.NS,281.521387,283.470097,277.623940,278.349060,12182400
4,2018-01-01,POWERGRID.NS,72.509389,73.492939,72.509389,72.928307,6388017


In [18]:
print(prices_long.columns)

Index(['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume'], dtype='str', name='Price')


In [19]:
print(prices_long.shape)

(42860, 7)


In [20]:
prices_long["Date"] = pd.to_datetime(prices_long["Date"])

prices_long = prices_long.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

prices_long.head()

Price,Date,Ticker,Open,High,Low,Close,Volume
0,2018-01-01,ADANIENT.NS,88.915500,91.087447,88.486475,89.049576,7745247
1,2018-01-02,ADANIENT.NS,89.076381,89.773550,86.958070,87.896561,4977129
2,2018-01-03,ADANIENT.NS,87.896563,96.664767,87.896563,94.492828,39032205
3,2018-01-04,ADANIENT.NS,94.385570,99.319362,93.393444,98.300423,22778210
4,2018-01-05,ADANIENT.NS,99.104834,104.199507,97.817759,103.582787,21144683


In [21]:
print(prices_long.dtypes)

Price
Date      datetime64[s]
Ticker              str
Open            float64
High            float64
Low             float64
Close           float64
Volume            int64
dtype: object


In [22]:
prices_long["return_5d"] = (
    prices_long.groupby("Ticker")["Close"]
    .pct_change(5)
)

In [23]:
prices_long[["Date", "Ticker", "Close", "return_5d"]].head(15)

Price,Date,Ticker,Close,return_5d
0,2018-01-01,ADANIENT.NS,89.049576,NaN
1,2018-01-02,ADANIENT.NS,87.896561,NaN
2,2018-01-03,ADANIENT.NS,94.492828,NaN
3,2018-01-04,ADANIENT.NS,98.300423,NaN
4,2018-01-05,ADANIENT.NS,103.582787,NaN
5,2018-01-08,ADANIENT.NS,105.352516,0.183077
6,2018-01-09,ADANIENT.NS,105.057571,0.195241
7,2018-01-10,ADANIENT.NS,106.612793,0.128263
8,2018-01-11,ADANIENT.NS,108.168007,0.100382
9,2018-01-12,ADANIENT.NS,107.229515,0.035206


In [24]:
prices_long["return_20d"] = (
    prices_long.groupby("Ticker")["Close"]
    .pct_change(20)
)

prices_long["return_60d"] = (
    prices_long.groupby("Ticker")["Close"]
    .pct_change(60)
)

In [25]:
prices_long[
    ["Date", "Ticker", "Close", "return_5d", "return_20d", "return_60d"]
].head(70)

Price,Date,Ticker,Close,return_5d,return_20d,return_60d
0,2018-01-01,ADANIENT.NS,89.049576,NaN,NaN,NaN
1,2018-01-02,ADANIENT.NS,87.896561,NaN,NaN,NaN
2,2018-01-03,ADANIENT.NS,94.492828,NaN,NaN,NaN
3,2018-01-04,ADANIENT.NS,98.300423,NaN,NaN,NaN
4,2018-01-05,ADANIENT.NS,103.582787,NaN,NaN,NaN
...,...,...,...,...,...,...
65,2018-04-09,ADANIENT.NS,92.921043,0.076875,0.025565,-0.117999
66,2018-04-10,ADANIENT.NS,93.299934,0.071279,0.114514,-0.111916
67,2018-04-11,ADANIENT.NS,91.279221,0.123854,0.049045,-0.143825
68,2018-04-12,ADANIENT.NS,85.943283,-0.079473,0.003178,-0.205465


In [26]:
prices_long["daily_return"] = (
    prices_long.groupby("Ticker")["Close"]
    .pct_change()
)

In [27]:
prices_long[["Date", "Ticker", "Close", "daily_return"]].head(10)

Price,Date,Ticker,Close,daily_return
0,2018-01-01,ADANIENT.NS,89.049576,NaN
1,2018-01-02,ADANIENT.NS,87.896561,-0.012948
2,2018-01-03,ADANIENT.NS,94.492828,0.075046
3,2018-01-04,ADANIENT.NS,98.300423,0.040295
4,2018-01-05,ADANIENT.NS,103.582787,0.053737
5,2018-01-08,ADANIENT.NS,105.352516,0.017085
6,2018-01-09,ADANIENT.NS,105.057571,-0.002800
7,2018-01-10,ADANIENT.NS,106.612793,0.014804
8,2018-01-11,ADANIENT.NS,108.168007,0.014587
9,2018-01-12,ADANIENT.NS,107.229515,-0.008676


In [28]:
prices_long["volatility_20d"] = (
    prices_long.groupby("Ticker")["daily_return"]
    .rolling(20)
    .std()
    .reset_index(level=0, drop=True)
)

In [29]:
prices_long[
    ["Date", "Ticker", "daily_return", "volatility_20d"]
].head(30)

Price,Date,Ticker,daily_return,volatility_20d
0,2018-01-01,ADANIENT.NS,NaN,NaN
1,2018-01-02,ADANIENT.NS,-0.012948,NaN
2,2018-01-03,ADANIENT.NS,0.075046,NaN
3,2018-01-04,ADANIENT.NS,0.040295,NaN
4,2018-01-05,ADANIENT.NS,0.053737,NaN
5,2018-01-08,ADANIENT.NS,0.017085,NaN
6,2018-01-09,ADANIENT.NS,-0.002800,NaN
7,2018-01-10,ADANIENT.NS,0.014804,NaN
8,2018-01-11,ADANIENT.NS,0.014587,NaN
9,2018-01-12,ADANIENT.NS,-0.008676,NaN


In [30]:
prices_long["MA50"] = (
    prices_long.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(50).mean())
)

In [31]:
prices_long["MA200"] = (
    prices_long.groupby("Ticker")["Close"]
    .transform(lambda x: x.rolling(200).mean())
)

In [32]:
prices_long["price_vs_MA50"] = (
    prices_long["Close"] / prices_long["MA50"] - 1
)

prices_long["price_vs_MA200"] = (
    prices_long["Close"] / prices_long["MA200"] - 1
)

In [33]:
import ta

prices_long["RSI"] = (
    prices_long.groupby("Ticker")["Close"]
    .transform(lambda x: ta.momentum.RSIIndicator(x, window=14).rsi())
)

In [34]:
prices_long[["Ticker", "Date", "Close", "RSI"]].head(20)

Price,Ticker,Date,Close,RSI
0,ADANIENT.NS,2018-01-01,89.049576,NaN
1,ADANIENT.NS,2018-01-02,87.896561,NaN
2,ADANIENT.NS,2018-01-03,94.492828,NaN
3,ADANIENT.NS,2018-01-04,98.300423,NaN
4,ADANIENT.NS,2018-01-05,103.582787,NaN
5,ADANIENT.NS,2018-01-08,105.352516,NaN
6,ADANIENT.NS,2018-01-09,105.057571,NaN
7,ADANIENT.NS,2018-01-10,106.612793,NaN
8,ADANIENT.NS,2018-01-11,108.168007,NaN
9,ADANIENT.NS,2018-01-12,107.229515,NaN


In [35]:
prices_long["volume_change_20d"] = (
    prices_long.groupby("Ticker")["Volume"]
    .pct_change(20)
)

In [36]:
prices_long["MACD"] = (
    prices_long.groupby("Ticker")["Close"]
    .transform(
        lambda x: ta.trend.MACD(x).macd()
    )
)

In [37]:
prices_long["MACD_signal"] = (
    prices_long.groupby("Ticker")["Close"]
    .transform(
        lambda x: ta.trend.MACD(x).macd_signal()
    )
)

In [38]:
prices_long["MACD_hist"] = (
    prices_long["MACD"] - prices_long["MACD_signal"]
)

In [39]:
prices_long["BB_high"] = (
    prices_long.groupby("Ticker")["Close"]
    .transform(
        lambda x: ta.volatility.BollingerBands(x, window=20, window_dev=2).bollinger_hband()
    )
)

prices_long["BB_low"] = (
    prices_long.groupby("Ticker")["Close"]
    .transform(
        lambda x: ta.volatility.BollingerBands(x, window=20, window_dev=2).bollinger_lband()
    )
)

In [40]:
prices_long["BB_position"] = (
    (prices_long["Close"] - prices_long["BB_low"])
    / (prices_long["BB_high"] - prices_long["BB_low"])
)

In [41]:
prices_long["ATR"] = (
    prices_long.groupby("Ticker")
    .apply(
        lambda x: ta.volatility.AverageTrueRange(
            high=x["High"],
            low=x["Low"],
            close=x["Close"],
            window=14
        ).average_true_range()
    )
    .reset_index(level=0, drop=True)
)

In [42]:
prices_long["ATR_pct"] = (
    prices_long["ATR"] / prices_long["Close"]
)

In [43]:
prices_long["future_return_5d"] = (
    prices_long.groupby("Ticker")["Close"]
    .shift(-5) / prices_long["Close"] - 1
)

In [44]:
print(prices_long[["Ticker", "Date", "Close", "future_return_5d"]].head(20))

Price       Ticker       Date       Close  future_return_5d
0      ADANIENT.NS 2018-01-01   89.049576          0.183077
1      ADANIENT.NS 2018-01-02   87.896561          0.195241
2      ADANIENT.NS 2018-01-03   94.492828          0.128263
3      ADANIENT.NS 2018-01-04   98.300423          0.100382
4      ADANIENT.NS 2018-01-05  103.582787          0.035206
5      ADANIENT.NS 2018-01-08  105.352516          0.008908
6      ADANIENT.NS 2018-01-09  105.057571         -0.006636
7      ADANIENT.NS 2018-01-10  106.612793          0.028924
8      ADANIENT.NS 2018-01-11  108.168007         -0.031978
9      ADANIENT.NS 2018-01-12  107.229515         -0.001250
10     ADANIENT.NS 2018-01-15  106.291008          0.003532
11     ADANIENT.NS 2018-01-16  104.360405          0.050360
12     ADANIENT.NS 2018-01-17  109.696411          0.004644
13     ADANIENT.NS 2018-01-18  104.709000          0.098079
14     ADANIENT.NS 2018-01-19  107.095428          0.068353
15     ADANIENT.NS 2018-01-22  106.66640

In [45]:
print(prices_long["future_return_5d"].describe())

count    42760.000000
mean         0.003607
std          0.041080
min         -0.538131
25%         -0.018194
50%          0.002538
75%          0.024101
max          0.661416
Name: future_return_5d, dtype: float64


In [46]:
prices_long["target"] = (
    prices_long["future_return_5d"] > 0
).astype(int)

In [47]:
print(prices_long["target"].value_counts())

target
1    22798
0    20062
Name: count, dtype: int64


In [48]:
print(prices_long["target"].value_counts(normalize=True))

target
1    0.531918
0    0.468082
Name: proportion, dtype: float64


In [49]:
model_data = prices_long.dropna().copy()

print("Original rows:", len(prices_long))
print("Rows for ML:", len(model_data))
print("Columns:", model_data.columns.tolist())

Original rows: 42860
Rows for ML: 38780
Columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume', 'return_5d', 'return_20d', 'return_60d', 'daily_return', 'volatility_20d', 'MA50', 'MA200', 'price_vs_MA50', 'price_vs_MA200', 'RSI', 'volume_change_20d', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_high', 'BB_low', 'BB_position', 'ATR', 'ATR_pct', 'future_return_5d', 'target']


In [50]:
features = [
    "return_5d",
    "price_vs_MA50",
    "price_vs_MA200",
    "RSI",
    "volume_change_20d",
    "MACD",
    "MACD_signal",
    "MACD_hist",
    "BB_position",
    "ATR_pct"
]

X = model_data[features]
y = model_data["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeatures:")
print(X.columns.tolist())

X shape: (38780, 10)
y shape: (38780,)

Features:
['return_5d', 'price_vs_MA50', 'price_vs_MA200', 'RSI', 'volume_change_20d', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_position', 'ATR_pct']


In [51]:
model_data = model_data.sort_values(["Date", "Ticker"]).reset_index(drop=True)

In [52]:
split_date = model_data["Date"].quantile(0.80)

train_data = model_data[model_data["Date"] <= split_date]
test_data = model_data[model_data["Date"] > split_date]

print("Split date:", split_date)
print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

Split date: 2025-02-01 00:00:00
Training rows: 31040
Testing rows: 7740


In [53]:
X_train = train_data[features]
y_train = train_data["target"]

X_test = test_data[features]
y_test = test_data["target"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (31040, 10)
y_train: (31040,)
X_test: (7740, 10)
y_test: (7740,)


In [55]:
print("X_train columns:")
print(X_train.columns.tolist())

print("\nX_test columns:")
print(X_test.columns.tolist())

X_train columns:
['return_5d', 'price_vs_MA50', 'price_vs_MA200', 'RSI', 'volume_change_20d', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_position', 'ATR_pct']

X_test columns:
['return_5d', 'price_vs_MA50', 'price_vs_MA200', 'RSI', 'volume_change_20d', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_position', 'ATR_pct']


In [56]:
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (31040, 10)
X_test shape: (7740, 10)


In [58]:
print("NaN in X_train:", X_train.isna().sum().sum())
print("NaN in X_test:", X_test.isna().sum().sum())

NaN in X_train: 0
NaN in X_test: 0


In [59]:
import numpy as np

print("Inf in X_train:", np.isinf(X_train).sum().sum())
print("Inf in X_test:", np.isinf(X_test).sum().sum())

Inf in X_train: 0
Inf in X_test: 99


In [60]:
import numpy as np

inf_counts = np.isinf(X_test).sum()

print(inf_counts)

Price
return_5d             0
price_vs_MA50         0
price_vs_MA200        0
RSI                   0
volume_change_20d    99
MACD                  0
MACD_signal           0
MACD_hist             0
BB_position           0
ATR_pct               0
dtype: int64


In [61]:
print(
    model_data[
        ~np.isfinite(model_data["volume_change_20d"])
    ][["Ticker", "Date", "Volume", "volume_change_20d"]]
)

Price         Ticker       Date    Volume  volume_change_20d
32020    ADANIENT.NS 2025-04-21    782559                inf
32021  ASIANPAINT.NS 2025-04-21    541913                inf
32022    AXISBANK.NS 2025-04-21  15166580                inf
32023  BAJFINANCE.NS 2025-04-21   7824150                inf
32024  BHARTIARTL.NS 2025-04-21   3501123                inf
...              ...        ...       ...                ...
38355        SBIN.NS 2026-07-24  13797053                inf
38356   SUNPHARMA.NS 2026-07-24   1578929                inf
38357         TCS.NS 2026-07-24   3211699                inf
38358       TITAN.NS 2026-07-24    513858                inf
38359       WIPRO.NS 2026-07-24  13282944                inf

[99 rows x 4 columns]


In [62]:
print(
    model_data[
        ~np.isfinite(model_data["volume_change_20d"])
    ]["Volume"].describe()
)

count    9.900000e+01
mean     1.477578e+07
std      1.870594e+07
min      2.086450e+05
25%      1.997753e+06
50%      9.817000e+06
75%      1.708818e+07
max      1.012994e+08
Name: Volume, dtype: float64


In [63]:
prices_long["Volume_20d_ago"] = (
    prices_long.groupby("Ticker")["Volume"]
    .shift(20)
)

print(
    prices_long[
        ~np.isfinite(prices_long["volume_change_20d"])
    ][
        ["Ticker", "Date", "Volume", "Volume_20d_ago", "volume_change_20d"]
    ].head(20)
)

Price       Ticker       Date    Volume  Volume_20d_ago  volume_change_20d
0      ADANIENT.NS 2018-01-01   7745247             NaN                NaN
1      ADANIENT.NS 2018-01-02   4977129             NaN                NaN
2      ADANIENT.NS 2018-01-03  39032205             NaN                NaN
3      ADANIENT.NS 2018-01-04  22778210             NaN                NaN
4      ADANIENT.NS 2018-01-05  21144683             NaN                NaN
5      ADANIENT.NS 2018-01-08  15147523             NaN                NaN
6      ADANIENT.NS 2018-01-09   8653790             NaN                NaN
7      ADANIENT.NS 2018-01-10  21411711             NaN                NaN
8      ADANIENT.NS 2018-01-11  10238800             NaN                NaN
9      ADANIENT.NS 2018-01-12   9577781             NaN                NaN
10     ADANIENT.NS 2018-01-15   5453962             NaN                NaN
11     ADANIENT.NS 2018-01-16  13454300             NaN                NaN
12     ADANIENT.NS 2018-0

In [65]:
print("Zero volume 20 days ago:",
      (prices_long["Volume_20d_ago"] == 0).sum())

print("Infinite volume changes:",
      np.isinf(prices_long["volume_change_20d"]).sum())

Zero volume 20 days ago: 99
Infinite volume changes: 99


In [66]:
prices_long["volume_change_20d"] = prices_long["volume_change_20d"].replace(
    [np.inf, -np.inf],
    np.nan
)

In [67]:
print(
    "Infinite values:",
    np.isinf(prices_long["volume_change_20d"]).sum()
)

print(
    "NaN values:",
    prices_long["volume_change_20d"].isna().sum()
)

Infinite values: 0
NaN values: 499


In [68]:
model_data = prices_long.dropna().copy()

print("ML dataset shape:", model_data.shape)

ML dataset shape: (38681, 29)


In [64]:
print(
    (prices_long["Volume_20d_ago"] == 0).sum()
)

99


In [69]:
model_data = model_data.sort_values(
    ["Date", "Ticker"]
).reset_index(drop=True)

In [70]:
split_date = model_data["Date"].quantile(0.80)

train_data = model_data[
    model_data["Date"] <= split_date
]

test_data = model_data[
    model_data["Date"] > split_date
]

print("Split date:", split_date)
print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

Split date: 2025-01-28 00:00:00
Training rows: 30960
Testing rows: 7721


In [71]:
X_train = train_data[features]
y_train = train_data["target"]

X_test = test_data[features]
y_test = test_data["target"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (30960, 10)
X_test: (7721, 10)


In [72]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [73]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

print("Model trained successfully!")

Model trained successfully!


In [74]:
y_pred = model.predict(X_test_scaled)

In [75]:
y_prob = model.predict_proba(X_test_scaled)

In [76]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.5016189612744463


In [77]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(cm)

[[  58 3809]
 [  39 3815]]


In [78]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.60      0.01      0.03      3867
           1       0.50      0.99      0.66      3854

    accuracy                           0.50      7721
   macro avg       0.55      0.50      0.35      7721
weighted avg       0.55      0.50      0.35      7721



In [79]:
print(y_train.value_counts(normalize=True))
print()
print(y_test.value_counts(normalize=True))

target
1    0.545801
0    0.454199
Name: proportion, dtype: float64

target
0    0.500842
1    0.499158
Name: proportion, dtype: float64


In [80]:
coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_[0]
})

coefficients = coefficients.sort_values(
    "Coefficient",
    ascending=False
)

print(coefficients)

             Feature  Coefficient
8        BB_position     0.088154
9            ATR_pct     0.014249
2     price_vs_MA200     0.009077
6        MACD_signal     0.004623
4  volume_change_20d    -0.000529
0          return_5d    -0.000550
1      price_vs_MA50    -0.007465
5               MACD    -0.008854
7          MACD_hist    -0.043073
3                RSI    -0.092136


In [81]:
prob_up = model.predict_proba(X_test_scaled)[:, 1]

print(pd.Series(prob_up).describe())

count    7721.000000
mean        0.545648
std         0.020098
min         0.335751
25%         0.538161
50%         0.545286
75%         0.552646
max         0.734708
dtype: float64


In [82]:
prob_up = model.predict_proba(X_test_scaled)[:, 1]

In [83]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, prob_up)

print("ROC-AUC:", auc)

ROC-AUC: 0.49195043714133224


In [84]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [85]:
rf_pred = rf_model.predict(X_test)

print(rf_pred[:20])

[1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 0 1 1 1]


In [86]:
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print(rf_prob[:20])

[0.60459421 0.58221144 0.56495072 0.54147307 0.56312472 0.57435916
 0.57874991 0.57331537 0.53681819 0.55029897 0.4708567  0.54592037
 0.55962316 0.55726543 0.55689471 0.57082981 0.48287192 0.56981331
 0.53437288 0.54581643]


In [87]:
from sklearn.metrics import accuracy_score

rf_accuracy = accuracy_score(y_test, rf_pred)

print("Random Forest Accuracy:", rf_accuracy)

Random Forest Accuracy: 0.5009713767646677


In [88]:
rf_auc = roc_auc_score(y_test, rf_prob)

print("Random Forest ROC-AUC:", rf_auc)

Random Forest ROC-AUC: 0.5146022207791529


In [89]:
rf_importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
})

rf_importance = rf_importance.sort_values(
    "Importance",
    ascending=False
)

print(rf_importance)

             Feature  Importance
2     price_vs_MA200    0.137993
7          MACD_hist    0.136266
9            ATR_pct    0.126139
1      price_vs_MA50    0.101215
6        MACD_signal    0.100842
5               MACD    0.095951
3                RSI    0.093187
8        BB_position    0.071712
0          return_5d    0.069819
4  volume_change_20d    0.066875


In [90]:
rf_importance = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
})

rf_importance = rf_importance.sort_values(
    "Importance",
    ascending=False
)

print(rf_importance)

             Feature  Importance
2     price_vs_MA200    0.137993
7          MACD_hist    0.136266
9            ATR_pct    0.126139
1      price_vs_MA50    0.101215
6        MACD_signal    0.100842
5               MACD    0.095951
3                RSI    0.093187
8        BB_position    0.071712
0          return_5d    0.069819
4  volume_change_20d    0.066875


In [91]:
from sklearn.metrics import confusion_matrix

rf_cm = confusion_matrix(y_test, rf_pred)

print(rf_cm)

[[ 440 3427]
 [ 426 3428]]


In [92]:
from sklearn.metrics import classification_report

print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.51      0.11      0.19      3867
           1       0.50      0.89      0.64      3854

    accuracy                           0.50      7721
   macro avg       0.50      0.50      0.41      7721
weighted avg       0.50      0.50      0.41      7721



In [93]:
from sklearn.ensemble import GradientBoostingClassifier

gb_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

print("Gradient Boosting trained successfully!")

Gradient Boosting trained successfully!


In [97]:
gb_model.fit(X_train, y_train)

,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (i

In [98]:
gb_pred = gb_model.predict(X_test)

print(gb_pred[:20])

[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1]


In [99]:
accuracy_score(y_test, gb_pred)

0.5083538401761429

In [100]:
gb_prob = gb_model.predict_proba(X_test)[:, 1]

In [101]:
gb_auc = roc_auc_score(y_test, gb_prob)

print("Gradient Boosting ROC-AUC:", gb_auc)

Gradient Boosting ROC-AUC: 0.5190507640596271


In [102]:
from sklearn.metrics import classification_report

print(classification_report(y_test, gb_pred))

              precision    recall  f1-score   support

           0       0.54      0.12      0.19      3867
           1       0.50      0.90      0.65      3854

    accuracy                           0.51      7721
   macro avg       0.52      0.51      0.42      7721
weighted avg       0.52      0.51      0.42      7721



In [103]:
print(classification_report(y_test, gb_pred))

              precision    recall  f1-score   support

           0       0.54      0.12      0.19      3867
           1       0.50      0.90      0.65      3854

    accuracy                           0.51      7721
   macro avg       0.52      0.51      0.42      7721
weighted avg       0.52      0.51      0.42      7721



In [105]:
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV

svm_base = SVC(
    C=1.0,
    kernel="rbf"
)

svm_model = CalibratedClassifierCV(
    svm_base,
    ensemble=False
)

svm_model.fit(X_train_scaled, y_train)

print("SVM trained successfully!")

SVM trained successfully!


In [106]:
svm_pred = svm_model.predict(X_test_scaled)

svm_prob = svm_model.predict_proba(X_test_scaled)[:, 1]

In [107]:
from sklearn.metrics import accuracy_score

svm_accuracy = accuracy_score(y_test, svm_pred)

print("SVM Accuracy:", svm_accuracy)

SVM Accuracy: 0.49915814013728793


In [108]:
svm_auc = roc_auc_score(y_test, svm_prob)

print("SVM ROC-AUC:", svm_auc)

SVM ROC-AUC: 0.4947046375536135


In [109]:
from sklearn.neighbors import KNeighborsClassifier

knn_model = KNeighborsClassifier(
    n_neighbors=25,
    weights="distance"
)

knn_model.fit(X_train_scaled, y_train)

print("KNN trained successfully!")

KNN trained successfully!


In [110]:
knn_pred = knn_model.predict(X_test_scaled)
knn_prob = knn_model.predict_proba(X_test_scaled)[:, 1]

In [111]:
knn_pred = knn_model.predict(X_test_scaled)

print(knn_pred[:20])

[0 1 1 1 1 1 1 1 1 0 0 1 1 1 0 1 0 0 0 1]


In [112]:
knn_prob = knn_model.predict_proba(X_test_scaled)[:, 1]

print(knn_prob[:20])

[0.36094457 0.78775518 0.57729543 0.54967784 0.66059893 0.64913493
 0.71590508 0.70079195 0.59946511 0.44359132 0.2791638  0.61587743
 0.57969878 0.54801423 0.46974613 0.67812938 0.42446982 0.48912265
 0.31960027 0.5637838 ]


In [113]:
knn_accuracy = accuracy_score(y_test, knn_pred)

knn_auc = roc_auc_score(y_test, knn_prob)

print("KNN Accuracy:", knn_accuracy)
print("KNN ROC-AUC:", knn_auc)

KNN Accuracy: 0.4946250485688382
KNN ROC-AUC: 0.5001201066762
